<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_02_Linear_Models%2C_MLP%2C_Backprop_and_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 2: Linear Models, Optimization, and Backpropagation


Instructor: John Sipple

### Overview

This tutorial progresses from the foundational analytical solutions of linear models to the iterative optimization methods that power deep neural networks. It serves as a rigorous bridge between classical statistical learning and modern deep learning architectures. Rather than relying on high-level abstractions, this notebook challenges you to manually implement the mathematical engines behind these models, from calculating closed-form solutions for linear regression to coding the chain rule for backpropagation from scratch. By validating manual gradient calculations against JAX's powerful AutoDiff engine, you will gain a deep, mechanical understanding of how neural networks learn and optimize.

### What it Explores:

* **Simple Linear Regression (Least Squares)**: Formulating the Residual Sum of Squares (RSS) and solving for weights analytically using the Normal Equation. It explores the process of generating synthetic data with additive Gaussian noise to validate the recovery of true parameters against learned parameters.


* **Linear Classification via Ridge Regression**: Repurposing least squares for multi-class classification using one-hot encoding and introducing L2 regularization (Ridge) to prevent overfitting and ensure invertibility. This section also explores evaluating performance by manually computing critical metrics like Precision, Recall, and F1 Score, alongside plotting confusion matrices.


* **Multilayer Perceptrons (MLP)**: Designing a non-linear feedforward neural network architecture with two hidden layers, ReLU activations, and a Softmax output layer for probability distributions. It includes implementing a custom visualization function to graphically map the network's architectural nodes and connection edges.


* **Backpropagation and Optimization**: The core of the tutorial involves mathematically deriving the Jacobian for Cross-Entropy loss combined with L2 regularization and applying the chain rule backward through the network layers. It explores manually implementing these gradient calculations, comparing them directly against JAX's `grad` function for rigorous verification, and executing a custom training loop utilizing Gradient Descent.

In [ ]:
import jax
import jax.numpy as jnp
from jax import random, grad, vmap, jit
import matplotlib.pyplot as plt
import seaborn as sns

# Set precision to 64-bit for rigorous gradient checking
jax.config.update("jax_enable_x64", True)

sns.set_theme(style="whitegrid")
print(f"JAX Version: {jax.__version__}")

## Part 1: Simple Linear Regression (Least Squares)

**Mathematical Formulation**

We assume a dataset $\mathscr{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$ where $\mathbf{x} \in \mathbb{R}^{D}$ and $y \in \mathbb{R}$.Our model is linear: $\hat{y} = \mathbf{x}^T \mathbf{w} + b$.

By augmenting the input $\mathbf{x}$ with a 1, we can absorb the bias $b$ into $\mathbf{w}$.We seek to minimize the Residual Sum of Squares (RSS)
:$$\mathscr{L}(\mathbf{w}) = ||\mathbf{X}\mathbf{w} - \mathbf{y}||_2^2$$

Taking the gradient $\nabla_w \mathscr{L} = 0$ yields the Normal Equation:$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

**Data Generation for Linear Regression**

To validate our models, we generate synthetic data where the ground truth is known. This allows us to compare our learned parameters $\hat{\mathbf{w}}$ directly against the true parameters $\mathbf{w}_{true}$.

**The Data Generation Process:**

1. **Inputs** ($\mathbf{X}$): Sampled from a standard normal distribution $\mathscr{N}(0, 1)$:  $\mathbf{x}_i \in \mathbb{R}^{10}$

2. **True Weights** ($\mathbf{w}_{true}$): Random vector sampled from $\mathscr{N}(0, 1)$.

3. **Noise** ($\epsilon$): Additive Gaussian noise to simulate real-world imperfection.$\epsilon_i \sim \mathscr{N}(0, 0.1)$

**The Model**:

$$y_i = \mathbf{x}_i^T \mathbf{w}_{true} + \epsilon_i$$

We generate **1,000 training samples** and **200 test samples**. The goal is to recover $\mathbf{w}_{true}$ as closely as possible using only the noisy observations $(\mathbf{X}, \mathbf{y})$.


In [ ]:
def generate_regression_data(key, n_samples=100, input_dim=10):
    k1, k2, k3 = random.split(key, 3)
    X = random.normal(k1, (n_samples, input_dim))
    true_w = random.normal(k2, (input_dim, 1))
    noise = 0.1 * random.normal(k3, (n_samples, 1))
    y = X @ true_w + noise
    return X, y, true_w

# 1. Data Generation
key = random.PRNGKey(42)
input_dim = 10
X_train, y_train, w_true = generate_regression_data(key, n_samples=10000, input_dim=input_dim)
X_test, y_test, _ = generate_regression_data(random.PRNGKey(43), n_samples=500, input_dim=input_dim)

# 2. Analytical Solution (Normal Equation)
def solve_least_squares(X, y):
    # w* = (X^T X)^-1 X^T y
    # Note: In production, jnp.linalg.solve is numerically more stable than inv()
    return jnp.linalg.solve(X.T @ X, X.T @ y)

w_ls = solve_least_squares(X_train, y_train)

# 3. Evaluation
mse_train = jnp.mean((X_train @ w_ls - y_train)**2)
mse_test = jnp.mean((X_test @ w_ls - y_test)**2)

print(f"Least Squares Regression Results:")
print(f"Weights Norm (True): {jnp.linalg.norm(w_true):.4f}")
print(f"Weights Norm (Est):  {jnp.linalg.norm(w_ls):.4f}")
print(f"Train MSE: {mse_train:.6f} | Test MSE: {mse_test:.6f}")

##Part 2: Linear Classification via Ridge Regression

**Mathematical Formulation**

Although Classification is typically solved via Logistic Regression (minimizing Cross-Entropy), we can formulate it as a Least Squares problem (often called the Linear Least Squares Classifier).

Given $K=3$ classes, we use **One-Hot Encoding** for targets $\mathbf{Y} \in \mathbb{R}^{N \times K}$. We solve for $\mathbf{W} \in \mathbb{R}^{D \times K}$.

To prevent overfitting and ensure invertibility, we add **L2 Regularization** (Ridge Regression):$$\mathscr{L}(\mathbf{W}) = ||\mathbf{X}\mathbf{W} - \mathbf{Y}||_F^2 + \lambda ||\mathbf{W}||_F^2$$

The closed-form solution is:$$\mathbf{W}^* = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^T \mathbf{Y}$$

For prediction, we apply a Sigmoid (or Softmax) to interpret outputs as probabilities, though the training minimized MSE.

In [ ]:
def generate_classification_data(key, n_samples=1000, input_dim=10, n_classes=3):
    k1, k2 = random.split(key)
    X = random.normal(k1, (n_samples, input_dim))
    # Generate random linear weights to determine class
    W_gen = random.normal(k2, (input_dim, n_classes))
    logits = X @ W_gen
    y_indices = jnp.argmax(logits, axis=1)
    # One-hot encode
    Y = jax.nn.one_hot(y_indices, n_classes)
    return X, Y

# 1. Data Generation
key = random.PRNGKey(101)
n_classes = 3
X_train_c, Y_train_c = generate_classification_data(key, n_samples=10000, n_classes=n_classes)
X_test_c, Y_test_c = generate_classification_data(random.PRNGKey(102), n_samples=500, n_classes=n_classes)

# 2. Ridge Regression Solution (L2 Regularized Least Squares)
def solve_ridge_regression(X, Y, lmbda=0.1):
    D = X.shape[1]
    I = jnp.eye(D)
    # W* = (X^T X + lambda*I)^-1 X^T Y
    return jnp.linalg.solve(X.T @ X + lmbda * I, X.T @ Y)

W_ridge = solve_ridge_regression(X_train_c, Y_train_c, lmbda=0.2)

# 3. Evaluation (Accuracy)
def predict_accuracy(X, Y_true, W):
    logits = X @ W
    # We can apply sigmoid/softmax here for probability interpretation
    probs = jax.nn.softmax(logits)
    preds = jnp.argmax(probs, axis=1)
    true_indices = jnp.argmax(Y_true, axis=1)
    return jnp.mean(preds == true_indices)

acc_train = predict_accuracy(X_train_c, Y_train_c, W_ridge)
acc_test = predict_accuracy(X_test_c, Y_test_c, W_ridge)

print(f"Ridge Classification Accuracy:")
print(f"Train: {acc_train*100:.2f}% | Test: {acc_test*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix

def compute_classification_metrics(Y_true_onehot, Y_pred_indices, class_labels=None):
    """
    Computes per-class Accuracy and F1 Score manually.

    Args:
        Y_true_onehot (jnp.array): Ground truth (N, num_classes)
        Y_pred_indices (jnp.array): Predicted class indices (N,)
        class_labels (list): Optional list of class names
    """
    y_true_indices = jnp.argmax(Y_true_onehot, axis=1)
    n_classes = Y_true_onehot.shape[1]

    if class_labels is None:
        class_labels = [f"Class {i}" for i in range(n_classes)]

    print(f"{'Class':<10} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1 Score':<10}")
    print("-" * 60)

    f1_scores = []

    for c in range(n_classes):
        # Create binary masks for the current class
        true_c = (y_true_indices == c)
        pred_c = (Y_pred_indices == c)

        # True Positives, False Positives, False Negatives
        tp = jnp.sum(true_c & pred_c)
        fp = jnp.sum((~true_c) & pred_c)
        fn = jnp.sum(true_c & (~pred_c))

        # Metrics
        # Accuracy per class: (TP + TN) / Total is often misleading in multi-class.
        # Here we report Recall (TP / (TP+FN)) which is "Accuracy on this class's samples"
        # but technically we can compute binary accuracy if desired.

        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
        f1_scores.append(f1)

        # 'Accuracy' for a specific class usually refers to Recall (Sensitivity)
        # i.e., "Of all real Class X, how many did we get right?"
        print(f"{class_labels[c]:<10} | {recall:.4f}     | {precision:.4f}     | {recall:.4f}     | {f1:.4f}")

    print("-" * 60)
    print(f"Macro Average F1 Score: {jnp.mean(jnp.array(f1_scores)):.4f}")

    # Overall Accuracy
    overall_acc = jnp.mean(y_true_indices == Y_pred_indices)
    print(f"Overall Global Accuracy:  {overall_acc:.4f}")


def plot_confusion_matrix(Y_true, predictions, title="Confusion Matrix"):
    """
    Plots a confusion matrix using Seaborn.

    Args:
        Y_true (array): One-hot encoded true labels.
        predictions (array): Predicted class indices.
    """
    # Convert one-hot true labels to indices
    y_true_indices = jnp.argmax(Y_true, axis=1)

    # Compute confusion matrix
    cm = confusion_matrix(y_true_indices, predictions)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=[f'Class {i}' for i in range(cm.shape[1])],
                yticklabels=[f'Class {i}' for i in range(cm.shape[0])])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.show()

# --- Visualize Part B Results ---
# Generate predictions for the test set using Ridge weights
logits_ridge = X_test_c @ W_ridge
preds_ridge = jnp.argmax(jax.nn.softmax(logits_ridge), axis=1)

plot_confusion_matrix(Y_test_c, preds_ridge, title="Part 2: Ridge Classification Confusion Matrix")

compute_classification_metrics(Y_test_c, preds_ridge)

## Part 3: Neural Network Architecture

We extend the previous models to a feedforward Neural Network with two hidden layers.

**Architecture**:

1. **Input**: $\mathbf{x} \in \mathbb{R}^{10}$

2. **Hidden 1**: $N_1 = 32$ units, ReLU activation.

3. **Hidden 2**: $N_2 = 16$ units, ReLU activation.

4. **Output**: $N_{out} = 3$ units, Softmax (for probability distribution).

**Forward Pass Equations**:

$$\mathbf{z}^{(1)} = \mathbf{x}\mathbf{W}^{(1)} + \mathbf{b}^{(1)} \\
\mathbf{a}^{(1)} = \text{ReLU}(\mathbf{z}^{(1)}) \\
\mathbf{z}^{(2)} = \mathbf{a}^{(1)}\mathbf{W}^{(2)} + \mathbf{b}^{(2)} \\
\mathbf{a}^{(2)} = \text{ReLU}(\mathbf{z}^{(2)}) \\
\mathbf{z}^{(3)} = \mathbf{a}^{(2)}\mathbf{W}^{(3)} + \mathbf{b}^{(3)} \\
\hat{\mathbf{y}} = \text{Softmax}(\mathbf{z}^{(3)})$$

In [ ]:
def init_mlp_params(key, layers=[10, 32, 16, 3]):
    params = []
    keys = random.split(key, len(layers))
    for i in range(len(layers) - 1):
        in_dim, out_dim = layers[i], layers[i+1]
        # Xavier/Glorot Initialization
        lim = jnp.sqrt(6.0 / (in_dim + out_dim))
        W = random.uniform(keys[i], (in_dim, out_dim), minval=-lim, maxval=lim)
        b = jnp.zeros(out_dim)
        params.append({'W': W, 'b': b})
    return params

def relu(x):
    return jnp.maximum(0, x)

def softmax(x):
    exps = jnp.exp(x - jnp.max(x, axis=1, keepdims=True)) # Stability fix
    return exps / jnp.sum(exps, axis=1, keepdims=True)

def forward(params, X):
    # Unpack layers
    activations = [X]

    # Hidden Layer 1
    z1 = activations[-1] @ params[0]['W'] + params[0]['b']
    a1 = relu(z1)
    activations.append(a1)

    # Hidden Layer 2
    z2 = activations[-1] @ params[1]['W'] + params[1]['b']
    a2 = relu(z2)
    activations.append(a2)

    # Output Layer (Logits)
    z3 = activations[-1] @ params[2]['W'] + params[2]['b']
    y_hat = softmax(z3)

    # We return intermediate values (z, a) for manual backprop
    cache = {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z3': z3, 'y_hat': y_hat}
    return y_hat, cache

In [ ]:
import matplotlib.pyplot as plt

def draw_neural_net(layer_sizes):
    """
    Visualizes a Feedforward Neural Network architecture.

    Args:
        layer_sizes (list): A list of integers representing the number of neurons in each layer.
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')

    n_layers = len(layer_sizes)
    v_spacing = 1.0 / float(max(layer_sizes))
    h_spacing = 1.0 / float(n_layers - 1)

    # Nodes
    for i, n in enumerate(layer_sizes):
        layer_top = v_spacing * (n - 1) / 2.0 + 0.5
        for m in range(n):
            # Coordinates
            x = i * h_spacing
            y = layer_top - m * v_spacing

            # Draw Circle
            circle = plt.Circle((x, y), v_spacing/4.0, color='w', ec='k', zorder=4)
            ax.add_artist(circle)

            # Annotation
            if i == 0:
                ax.text(x - 0.02, y, f"$x_{{{m+1}}}$", ha='right', va='center', fontsize=8)
            elif i == n_layers - 1:
                ax.text(x + 0.02, y, f"$\hat{{y}}_{{{m+1}}}$", ha='left', va='center', fontsize=12)

    # Edges (Connections)
    # We only draw a subset of edges for dense layers to avoid visual clutter
    for i, (n_a, n_b) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        layer_top_a = v_spacing * (n_a - 1) / 2.0 + 0.5
        layer_top_b = v_spacing * (n_b - 1) / 2.0 + 0.5

        for m in range(n_a):
            for o in range(n_b):
                x_a = i * h_spacing
                y_a = layer_top_a - m * v_spacing

                x_b = (i + 1) * h_spacing
                y_b = layer_top_b - o * v_spacing

                line = plt.Line2D([x_a, x_b], [y_a, y_b], c='k', alpha=0.1, lw=0.5)
                ax.add_artist(line)

    # Layer Labels
    layer_names = ["Input Layer\n(10 units)", "Hidden Layer 1\n(32 units, ReLU)",
                   "Hidden Layer 2\n(16 units, ReLU)", "Output Layer\n(3 units, Softmax)"]

    for i, label in enumerate(layer_names):
        x = i * h_spacing
        ax.text(x, 0.95, label, ha='center', va='bottom', fontsize=10, weight='bold')

    plt.title("MLP Architecture Visualization", fontsize=15)
    plt.tight_layout()
    plt.show()




##Part 4: Backpropagation & Optimization



###4.1 Mathematical Derivation of Gradients

We use the **Cross-Entropy Loss** with L2 Regularization:

$$J = -\sum_{k} y_k \log(\hat{y}_k) + \frac{\lambda}{2} \sum_l ||W^{(l)}||^2$$

**The Jacobian** of the loss with respect to the output logits $z^{(3)}$ (assuming Softmax + CrossEntropy) is simply:

$$\delta^{(3)} = \frac{\partial J}{\partial z^{(3)}} = \hat{\mathbf{y}} - \mathbf{y}$$

Using the Chain Rule, we propagate this error backwards:

**Layer 3 Gradients**:

$$\frac{\partial J}{\partial W^{(3)}} = (a^{(2)})^T \delta^{(3)} + \lambda W^{(3)}$$

$$\frac{\partial J}{\partial b^{(3)}} = \sum \delta^{(3)}$$

**Backprop to Hidden 2**:

$$\delta^{(2)} = (\delta^{(3)} (W^{(3)})^T) \odot \sigma'(z^{(2)})$$


Where $\sigma'$ is the derivative of ReLU (1 if $z>0$, else 0).


**Layer 2 Gradients**:

$$\frac{\partial J}{\partial W^{(2)}} = (a^{(1)})^T \delta^{(2)} + \lambda W^{(2)}$$

**Backprop to Hidden 1**:

$$\delta^{(1)} = (\delta^{(2)} (W^{(2)})^T) \odot \sigma'(z^{(1)})$$

**Layer 1 Gradients**:

$$\frac{\partial J}{\partial W^{(1)}} = (X)^T \delta^{(1)} + \lambda W^{(1)}$$

In [ ]:
def manual_backprop(params, X, Y, lmbda=0.01):
    # 1. Forward Pass
    y_hat, cache = forward(params, X)
    N = X.shape[0]

    # 2. Compute Deltas (Error Terms)
    # Output layer delta (Softmax + CrossEntropy combined derivative)
    delta3 = (y_hat - Y) / N  # Normalize by batch size

    # Backprop to Hidden 2
    # Derivative of ReLU: 1 if z > 0 else 0
    d_relu2 = (cache['z2'] > 0).astype(jnp.float64)
    delta2 = (delta3 @ params[2]['W'].T) * d_relu2

    # Backprop to Hidden 1
    d_relu1 = (cache['z1'] > 0).astype(jnp.float64)
    delta1 = (delta2 @ params[1]['W'].T) * d_relu1

    # 3. Compute Gradients for Weights and Biases
    grads = []

    # Layer 1
    dW1 = X.T @ delta1 + lmbda * params[0]['W']
    db1 = jnp.sum(delta1, axis=0)
    grads.append({'W': dW1, 'b': db1})

    # Layer 2
    dW2 = cache['a1'].T @ delta2 + lmbda * params[1]['W']
    db2 = jnp.sum(delta2, axis=0)
    grads.append({'W': dW2, 'b': db2})

    # Layer 3
    dW3 = cache['a2'].T @ delta3 + lmbda * params[2]['W']
    db3 = jnp.sum(delta3, axis=0)
    grads.append({'W': dW3, 'b': db3})

    return grads

# --- Verification with JAX AutoDiff ---
# Define scalar loss function for JAX
def loss_fn(params, X, Y, lmbda=0.01):
    y_hat, _ = forward(params, X)
    # Cross Entropy
    loss = -jnp.mean(jnp.sum(Y * jnp.log(y_hat + 1e-9), axis=1))
    # L2 Regularization
    l2_loss = 0.5 * lmbda * sum([jnp.sum(p['W']**2) for p in params])
    return loss + l2_loss

# Initialize
layers = [10, 32, 16, 3]
params = init_mlp_params(random.PRNGKey(55), layers)

# Visualize the neural net
draw_neural_net(layers)

# Compute Manual Gradients
manual_grads = manual_backprop(params, X_train_c, Y_train_c)

# Compute AutoDiff Gradients
jax_grads_fn = jax.grad(loss_fn)(params, X_train_c, Y_train_c,  )

# Compare a specific weight gradient
print("Gradient Verification (Layer 1 W):")
print(f"Manual Top-left: {manual_grads[0]['W'][0,0]:.6f}")
print(f"JAX AutoDiff:    {jax_grads_fn[0]['W'][0,0]:.6f}")
diff = jnp.linalg.norm(manual_grads[0]['W'] - jax_grads_fn[0]['W'])
print(f"Euclidean Difference: {diff:.6e}")

###4.2. Training Loop with Gradient Descent

We implement the update rule:

$$\theta_{new} = \theta_{old} - \eta \cdot \nabla_{\theta} J$$

In [ ]:
def cross_entropy_loss(params, X, Y):
    y_hat, _ = forward(params, X)
    return -jnp.mean(jnp.sum(Y * jnp.log(y_hat + 1e-9), axis=1))

def train_mlp(X_train, Y_train, X_test, Y_test, layers, epochs=500, lr=0.01, lmbda=0.001):
    params = init_mlp_params(random.PRNGKey(99), layers = layers)

    history = {'train_loss': [], 'test_loss': []}

    for i in range(epochs):
        # 1. Compute Gradients (Manual implementation)
        grads = manual_backprop(params, X_train, Y_train, lmbda)

        # 2. Update Weights (Gradient Descent)
        for l in range(len(params)):
            params[l]['W'] -= lr * grads[l]['W']
            params[l]['b'] -= lr * grads[l]['b']

        # 3. Logging
        if i % 10 == 0:
            train_loss = cross_entropy_loss(params, X_train, Y_train)
            test_loss = cross_entropy_loss(params, X_test, Y_test)
            history['train_loss'].append(train_loss)
            history['test_loss'].append(test_loss)

    return params, history

# Run Training
epochs = 500
lr = 0.006
lmbda = 0.001
layers = [10, 128, 64, 3]

final_params, history = train_mlp(X_train_c, Y_train_c, X_test_c, Y_test_c, layers = layers,
                                  epochs = epochs, lr = lr, lmbda=lmbda)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss (Cross Entropy)')
plt.plot(history['test_loss'], label='Test Loss (Cross Entropy)', linestyle='--')
plt.xlabel('Iterations (x10)')
plt.ylabel('Loss')
plt.title('Training Dynamics: MLP with Manual Backprop')
plt.legend()
plt.show()

In [ ]:
# --- Visualize Part D Results ---

# 1. Forward pass on test set using final trained parameters
y_hat_mlp, _ = forward(final_params, X_test_c)

# 2. Get class indices
preds_mlp = jnp.argmax(y_hat_mlp, axis=1)

# 3. Plot and view performance
plot_confusion_matrix(Y_test_c, preds_mlp, title="Part D: MLP Classification Confusion Matrix")

compute_classification_metrics(Y_test_c, preds_mlp)